# 00 · Local environment and model serving
**Complete before class.** This notebook works on its own and contains no practical answers.


**First open:** install Python 3.12, save this notebook in a writable folder, then run the commands below in a **terminal**, not a Python cell. If you downloaded the student pack, extract it and work in its folder.

| Platform | Terminal commands, one at a time |
|---|---|
| Windows | `py -3.12 -m venv .venv`<br>`.\.venv\Scripts\python.exe -m pip install "jupyterlab>=4,<5" "ipywidgets>=8.1,<9" "ipykernel>=6,<8"`<br>`.\.venv\Scripts\python.exe -m jupyterlab` |
| macOS / Linux | `python3 -m venv .venv`<br>`.venv/bin/python -m pip install 'jupyterlab>=4,<5' 'ipywidgets>=8.1,<9' 'ipykernel>=6,<8'`<br>`.venv/bin/python -m jupyterlab` |

Open this notebook in the browser and select the Python kernel from this environment. Existing course environments are fine if the checks below pass. A GPU and paid API are unnecessary. Use a local Jupyter kernel: a hosted notebook's `localhost` is not your laptop.

## 1 · Python and notebook dependencies
Run top to bottom. The environment check does not install or replace Python. A wrong kernel must be changed in Jupyter.

In [ ]:
import sys, json, platform, importlib.util, sqlite3, subprocess, shutil, os, time
from pathlib import Path
from urllib import request, error
from urllib.parse import urlparse

ROOT = Path.cwd().resolve()
if sys.version_info < (3, 10):
    raise RuntimeError("Python 3.10+ is required; use Python 3.12 for the course environment.")
print("Python:", sys.version.split()[0], "| executable:", sys.executable)
print("Working folder:", ROOT)
print("SQLite:", sqlite3.sqlite_version)
with sqlite3.connect(":memory:") as conn:
    assert conn.execute("SELECT 1").fetchone()[0] == 1

In [ ]:
# Install into THIS kernel only if the widget dependency is absent.
if importlib.util.find_spec("ipywidgets") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "ipywidgets>=8.1,<9"], check=True)
    print("If the next widget does not render, restart the kernel and run again.")
import ipywidgets as widgets
from IPython.display import display
widget_test = widgets.IntSlider(description="Widget test")
display(widget_test)
print("Move the slider. A plain text representation is not a working widget UI.")

## 2 · Install Ollama once
Ollama is a separate application; `pip install ollama` alone does not install its server.

- **Windows:** use the installer from [ollama.com/download/windows](https://ollama.com/download/windows). Launch Ollama from the Start menu.
- **macOS:** use [ollama.com/download/mac](https://ollama.com/download/mac). Open the installed application.
- **Linux:** follow the [official installation page](https://docs.ollama.com/linux). Its installation command is `curl -fsSL https://ollama.com/install.sh | sh`; run it in a terminal. If its service is not running, `ollama serve` starts the server.

The notebook checks the server before attempting to launch one. A desktop installation usually already runs it. After installing, restart Jupyter if its terminal cannot find the new `ollama` command.

Official platform requirements: [Windows](https://docs.ollama.com/windows), [macOS](https://docs.ollama.com/macos), [Linux](https://docs.ollama.com/linux). Choose **local models**; this exercise needs no cloud account or API key.

In [ ]:
# Keep this local endpoint. The application later reads the same configuration.
BASE_URL = "http://127.0.0.1:11434"
MODEL_NAME = "llama3.2:1b"
CPU_ONLY = True
THINK = None 
START_SERVER_IF_NEEDED = True
DOWNLOAD_IF_MISSING = True

parsed = urlparse(BASE_URL)
assert parsed.scheme == "http" and parsed.hostname in {"localhost", "127.0.0.1", "::1"}
assert not MODEL_NAME.endswith(":cloud"), "Select a local model tag."

# Re-running this cell keeps ownership of any server this notebook started.
if "started_server" not in globals():
    started_server = None

def api_json(path, payload=None, timeout=5):
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    req = request.Request(BASE_URL.rstrip("/") + path, data=data,
                          headers={"Content-Type": "application/json"})
    try:
        with request.urlopen(req, timeout=timeout) as response:
            return json.load(response)
    except error.HTTPError as exc:
        detail = exc.read(2000).decode("utf-8", errors="replace")
        raise RuntimeError(f"Ollama HTTP {exc.code}: {detail}") from exc

def server_available():
    try:
        return isinstance(api_json("/api/version").get("version"), str)
    except (OSError, RuntimeError, ValueError):
        return False

In [ ]:
if not server_available():
    cli = shutil.which("ollama")
    if not START_SERVER_IF_NEEDED or cli is None:
        raise RuntimeError("Open/install Ollama, or run 'ollama serve' in a terminal, then rerun this cell.")
    server_env = os.environ.copy()
    server_env["OLLAMA_HOST"] = parsed.netloc
    # Log output to a file; do not leave this notebook cell blocked by 'serve'.
    with (ROOT / "ollama_setup.log").open("ab") as log:
        started_server = subprocess.Popen([cli, "serve"], stdout=log,
                                          stderr=subprocess.STDOUT, env=server_env)
    for _ in range(20):
        if server_available():
            break
        if started_server.poll() is not None:
            break
        time.sleep(0.5)
if not server_available():
    raise RuntimeError("Server did not become ready. Inspect ollama_setup.log or the Ollama application logs.")
server_version = api_json("/api/version")["version"]
print("Ollama server:", server_version, "at", BASE_URL)
print("Started by this notebook:", started_server is not None and started_server.poll() is None)

## 3 · Download a small model before class
The default [Llama3.2:1b](https://ollama.com/library/llama3.2:1b) is a small baseline for the classification exercise, not a quality guarantee. Keep the instructor's agreed model tag for comparable results. It is not necessary to download a newer or larger model.

The next cell lists local models and downloads this tag only if absent. The first download needs internet and disk space. It may take several minutes; leave the cell running. The alternative terminal command is `ollama pull llama3.2:1b`.

Model files are stored by Ollama, outside the notebook package. They remain available after restarting Jupyter. Disk size is not the same as inference RAM usage.

In [ ]:
models = api_json("/api/tags").get("models", [])
installed_names = {m.get("name") for m in models}
print("Installed:", sorted(n for n in installed_names if n))
if MODEL_NAME not in installed_names:
    if not DOWNLOAD_IF_MISSING:
        raise RuntimeError(f"Download {MODEL_NAME} with 'ollama pull {MODEL_NAME}', then rerun.")
    req = request.Request(BASE_URL.rstrip("/") + "/api/pull",
        data=json.dumps({"model": MODEL_NAME, "stream": True}).encode("utf-8"),
        headers={"Content-Type": "application/json"})
    last_progress = None
    # NDJSON progress stream; timeout applies to socket I/O, not the entire download.
    with request.urlopen(req, timeout=120) as response:
        for line in response:
            if not line.strip():
                continue
            event = json.loads(line)
            if event.get("error"):
                raise RuntimeError(event["error"])
            percent = int(100 * event.get("completed", 0) / event["total"]) if event.get("total") else None
            progress = (event.get("status"), None if percent is None else percent // 10)
            if progress != last_progress:
                print(event.get("status"), "" if percent is None else f"{percent}%", flush=True)
                last_progress = progress
models = api_json("/api/tags").get("models", [])
selected = next((m for m in models if m.get("name") == MODEL_NAME), None)
if selected is None:
    raise RuntimeError("The requested exact model tag is still absent. Check /api/tags or 'ollama list'.")
print("Ready on disk:", selected["name"], "| digest:", selected.get("digest"))

## 4 · A real structured-output smoke test
This sends one HTTP request to your Ollama server. It tests serving, schema support and a completed response, not support-task accuracy.

The CPU setting sends `num_gpu=0`. After loading, inspect the process information (or `ollama ps`) to verify actual placement. The first call may include model-loading time. The configured socket timeout is 120 seconds, not a promised response latency.

In [ ]:
probe_schema = {"type":"object", "properties":{"ready":{"type":"boolean"}},
                "required":["ready"], "additionalProperties":False}
config = {"model":MODEL_NAME, "base_url":BASE_URL, "timeout_s":120,
          "output_mode":"schema", "num_ctx":2048, "num_predict":96,
          "cpu_only":CPU_ONLY, "think":THINK}
options = {"temperature":0, "num_ctx":config["num_ctx"], "num_predict":config["num_predict"]}
if CPU_ONLY:
    options["num_gpu"] = 0
payload = {"model":MODEL_NAME, "messages":[{"role":"user", "content":'Return only {"ready":true}.'}],
           "stream":False, "format":probe_schema, "options":options, "keep_alive":"5m"}
if THINK is not None:
    payload["think"] = THINK
start = time.perf_counter()
probe = api_json("/api/chat", payload, timeout=config["timeout_s"])
probe_elapsed_s = time.perf_counter() - start
if probe.get("done") is not True or probe.get("done_reason") == "length":
    raise RuntimeError("Model output was incomplete. Check model/thinking settings and output budget.")
probe_value = json.loads(probe["message"]["content"])
if probe_value != {"ready":True} or type(probe_value["ready"]) is not bool:
    raise RuntimeError(f"Unexpected probe output: {probe_value!r}")
print("Structured output:", probe_value)
print("Elapsed seconds:", round(probe_elapsed_s, 3))
print("Tokens:", {k:probe.get(k) for k in ("prompt_eval_count","eval_count")})

In [ ]:
processes = api_json("/api/ps").get("models", [])
loaded = next((m for m in processes if m.get("name") == MODEL_NAME or m.get("model") == MODEL_NAME), None)
cpu_verified = bool(loaded is not None and loaded.get("size_vram") == 0)
print("Loaded process:", loaded)
if CPU_ONLY and not cpu_verified:
    print("CPU placement was not confirmed. Inspect 'ollama ps' and report this to the teaching team.")
else:
    print("CPU placement confirmed." if CPU_ONLY else "Hardware placement recorded.")

## 5 · Save the configuration and readiness report
Move these two files with the notebook if you later extract the practical elsewhere. The practical reads `agentic_ai_config.json` from its root folder. Keep the model name, configuration and measured results together.

A working slider is a manual check: set the value below only after moving it successfully. A probe that passes does not establish model accuracy or the classroom latency target.

In [ ]:
WIDGET_DISPLAY_CONFIRMED = False  # Set True after you have moved the slider in section 1.

(ROOT / "agentic_ai_config.json").write_text(json.dumps(config, indent=2) + "\n", encoding="utf-8")
report = {"python":sys.version.split()[0], "platform":platform.system(),
          "sqlite_version":sqlite3.sqlite_version, "ollama_version":server_version,
          "model":MODEL_NAME, "model_digest":selected.get("digest"),
          "model_size_bytes":selected.get("size"), "config":config,
          "schema_probe_passed":True, "probe_elapsed_s":round(probe_elapsed_s,6),
          "cpu_placement_verified":cpu_verified, "widget_display_confirmed":WIDGET_DISPLAY_CONFIRMED}
report["ready_for_practical"] = bool(WIDGET_DISPLAY_CONFIRMED and (not CPU_ONLY or cpu_verified))
(ROOT / "setup_report.json").write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print("Saved configuration and setup_report.json")
print("Ready for practical:", report["ready_for_practical"])
print("If False, resolve the reported check before class; send the report to the teaching team if needed.")

## Troubleshooting

| Symptom | Next action |
|---|---|
| No Jupyter command / wrong Python | Launch with the environment's full Python path as shown at the top. |
| Widget appears as text | Check `ipywidgets` in this kernel and `jupyterlab_widgets` in the JupyterLab environment; restart kernel/browser. |
| Connection refused | Open Ollama, or run `ollama serve`; keep the kernel on the same computer. |
| Address already in use | A server may already be running. Check `/api/version`; do not start a second copy. |
| Model not found | Compare the exact `MODEL_NAME` with `/api/tags` or `ollama list`. |
| Download interrupted | Rerun the download cell or `ollama pull` after restoring the connection. |
| Unsupported OS / installation blocked | Check the official platform requirements and contact the teaching team before class. |
| Slow first call | Record loading and warm-call times separately; close other heavy processes. |
| Empty/truncated output | Keep the default non-thinking model, or explicitly disable supported thinking output; inspect the output limit. |
| Schema rejected | Update Ollama or report its version; do not silently label prompt-only output as schema-constrained. |
| CPU check uncertain | Inspect `ollama ps`; share the model tag, server version and setup report. |

To resume tomorrow: start Ollama, launch Jupyter from the same environment, open `01_Practical_1_Student.ipynb`. Existing models do not need downloading again.

In [ ]:
# Optional cleanup: stops ONLY a server launched by this notebook.
# Leave False while working on the practical. Desktop/system services are untouched.
STOP_NOTEBOOK_SERVER = False
if STOP_NOTEBOOK_SERVER and started_server is not None and started_server.poll() is None:
    started_server.terminate()
    started_server.wait(timeout=10)
    print("Stopped the notebook-owned server. Run section 2 again when needed.")

## Primary references
Checked 24 September 2026: [Ollama installation and local serving](https://docs.ollama.com/quickstart), [chat API](https://docs.ollama.com/api/chat), [structured output](https://docs.ollama.com/capabilities/structured-outputs), [model list](https://docs.ollama.com/api/tags), [model pull](https://docs.ollama.com/api/pull), [runtime options source](https://github.com/ollama/ollama/blob/main/api/types.go), [Python environments](https://docs.python.org/3/library/venv.html), [JupyterLab installation](https://jupyterlab.readthedocs.io/en/stable/getting_started/installation.html).